# 🔗 Notebook 04 — GNN Enrichment (GraphSAGE)
## Financial Representation Learning System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cliffordnwanna/FRL-SYSTEM/blob/main/notebooks/04_gnn_enrichment.ipynb)

**Purpose:** Train a GraphSAGE model to enrich customer embeddings with network-level signals from the bipartite graph.

**Why GraphSAGE over GCN:** GraphSAGE is inductive — it works on new customers (nodes) not seen during training. GCN breaks when you add new accounts. Production requires inductive learning.

**Training:** Unsupervised link prediction. No labels needed.

**Output:** `data/synthetic/enriched_embeddings.pkl`

**Runtime:** ~8 minutes on T4 GPU

## Setup

In [ ]:
# Clone the repository (if running on Colab)
import os

REPO_URL = "https://github.com/cliffordnwanna/FRL-SYSTEM.git"
REPO_DIR = "/content/FRL-SYSTEM"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
    print(f"✅ Repo cloned to {REPO_DIR}")
else:
    os.system(f"cd {REPO_DIR} && git pull")
    print(f"✅ Repo updated")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Mount Google Drive and restore artifacts from earlier notebooks/sessions
# This makes the pipeline resumable across disconnects, closed tabs, or a
# fresh runtime the next day — you don't need to keep one Colab tab open.
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DRIVE_DIR = "/content/drive/MyDrive/FRL-SYSTEM-artifacts"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs("data/synthetic", exist_ok=True)

restored = 0
for fname in os.listdir(DRIVE_DIR):
    src = os.path.join(DRIVE_DIR, fname)
    dst = os.path.join("data/synthetic", fname)
    if os.path.isfile(src):
        shutil.copy(src, dst)
        restored += 1
        print(f"Restored {fname} from Drive")
print(f"✅ Restored {restored} artifact(s) from Drive" if restored else "ℹ️ No prior artifacts found in Drive yet")

In [ ]:
import os, sys
os.chdir("/content/FRL-SYSTEM")
sys.path.insert(0, ".")

import pickle
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.gnn import GraphSAGEEncoder, train_gnn, enrich_embeddings, save_gnn
from src.graph_builder import load_graph
from src.encoder import load_embeddings

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")

## Step 1 — Load Graph and Embeddings

In [ ]:
graph_data = load_graph("data/synthetic/graph_data.pkl")
embeddings = load_embeddings("data/synthetic/customer_embeddings.pkl")

print(f"Graph nodes     : {graph_data['x'].shape[0]:,}")
print(f"Graph edges     : {graph_data['edge_index'].shape[1]:,}")
print(f"Node feature dim: {graph_data['x'].shape[1]}")
print(f"Customers       : {graph_data['n_customers']:,}")

## Step 2 — Train GraphSAGE

In [ ]:
model, history = train_gnn(graph_data, device=device)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history, color='#9B59B6', linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Link Prediction Loss")
plt.title("GraphSAGE Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("docs/results/04_gnn_loss.png", dpi=150, bbox_inches='tight')
plt.show()

## Step 3 — Extract Enriched Embeddings

In [ ]:
enriched = enrich_embeddings(model, graph_data, embeddings, device=device)

# Save
with open("data/synthetic/enriched_embeddings.pkl", "wb") as f:
    pickle.dump(enriched, f)
save_gnn(model, "data/synthetic/gnn.pt")
print("\n✅ Saved enriched_embeddings.pkl and gnn.pt")

## Step 4 — Compare Original vs Enriched

In [ ]:
# Sample 500 customers
sample_ids = list(enriched.keys())[:500]

orig_embs = np.array([embeddings[cid]["embedding"] for cid in sample_ids])
enr_embs  = np.array([enriched[cid]["embedding"] for cid in sample_ids])
archetypes = [enriched[cid]["archetype"] for cid in sample_ids]

# Cosine similarity within/across archetypes — original vs enriched
arch_arr = np.array(archetypes)
arch_labels = sorted(set(archetypes))

def cohesion(embs, archs):
    within, across = [], []
    norms = embs / (np.linalg.norm(embs, axis=1, keepdims=True) + 1e-8)
    sim = norms @ norms.T
    for i in range(len(embs)):
        for j in range(i+1, min(i+50, len(embs))):
            if archs[i] == archs[j]:
                within.append(sim[i,j])
            else:
                across.append(sim[i,j])
    return np.mean(within), np.mean(across)

w_orig, a_orig = cohesion(orig_embs, arch_arr)
w_enr,  a_enr  = cohesion(enr_embs, arch_arr)

print("Embedding Quality Comparison")
print("=" * 50)
print(f"{'':25s} {'Original':>12} {'Graph-Enriched':>14}")
print("-" * 50)
print(f"{'Within-archetype sim':25s} {w_orig:>12.4f} {w_enr:>14.4f}")
print(f"{'Across-archetype sim':25s} {a_orig:>12.4f} {a_enr:>14.4f}")
print(f"{'Separation ratio':25s} {w_orig/max(a_orig,0.001):>12.2f}x {w_enr/max(a_enr,0.001):>13.2f}x")
print("\nHigher separation ratio = better structured embedding space")

In [ ]:
# Persist this notebook's outputs to Drive so the next notebook — in this
# session or a brand new one tomorrow — can pick up where this left off.
import shutil, os
DRIVE_DIR = "/content/drive/MyDrive/FRL-SYSTEM-artifacts"
saved = 0
for fname in os.listdir("data/synthetic"):
    if fname.endswith((".csv", ".pkl", ".pt")):
        shutil.copy(os.path.join("data/synthetic", fname),
                    os.path.join(DRIVE_DIR, fname))
        saved += 1
print(f"✅ Saved {saved} artifact(s) to Drive: {DRIVE_DIR}")

## ✅ Notebook 04 Complete

**What was learned:**
- Graph-enriched 64-dim customer embeddings
- Each customer is now aware of their network neighbourhood
- Similar merchants → similar customers cluster more tightly
- Isolated customers (high churn risk) stand apart

**Next:** Run `05_downstream_tasks.ipynb` for segmentation, UMAP, anomaly detection, and next-product prediction.